In [1]:
import os
import sys
import pandas as pd
from dotenv import load_dotenv

# # 1. 定位项目根目录（Jupyter 兼容方式）
# from pathlib import Path

# # 1. 智能定位当前文件路径
# try:
#     # 脚本环境下有 __file__
#     current_path = Path(__file__).resolve()
# except NameError:
#     # Jupyter 环境下使用当前工作目录
#     current_path = Path.cwd()

# # 2. 向上跳级（假设根目录在当前文件向上两级）
# # .parent 代表上一级，.parents[1] 代表上两级
# PROJECT_ROOT = current_path.parents[0]

# print(f"项目根目录: {PROJECT_ROOT}")

# # try:
# #     # 脚本环境下有 __file__
# #     current_path = Path(__file__).resolve()
# # except NameError:
# #     # Jupyter 环境下使用当前工作目录
# #     current_path = Path.cwd()

# # # 2. 向上跳级（假设根目录在当前文件向上两级）
# # # .parent 代表上一级，.parents[1] 代表上两级
# # PROJECT_ROOT = current_path.parents[1]

# # 2. 将根目录加入系统路径，确保能 import core
# if PROJECT_ROOT not in sys.path:
#     sys.path.insert(0, PROJECT_ROOT)

# 1. 获取当前文件(docs)的绝对路径
current_dir = os.getcwd() 

# 2. 获取根目录 Alpha101 的路径 (即 docs 的上一级)
PROJECT_ROOT = os.path.dirname(current_dir)

# 3. 将根目录添加到 sys.path
if PROJECT_ROOT not in sys.path:
    # 使用 insert(0, ...) 确保优先搜索该路径
    sys.path.insert(0, PROJECT_ROOT)

# 4. 打印一下，确认 PROJECT_ROOT 是不是 Alpha101 的绝对路径
print(f"当前项目根目录: {PROJECT_ROOT}")

# 3. 导入你的 Alpha 因子函数
try:
    from core.alpha_factors import calculateAlpha002
    print("✓ 成功导入")
except ImportError as e:
    print(f"导入失败: {e}")


当前项目根目录: /Applications/obsidian repo/Obsidian Repo/Finance/Investment/Alpha101
✓ 成功导入


In [3]:
def sql_conn(query_template: str) -> pd.DataFrame:
    """
    连接数据库并执行查询
    """
    import sqlite3
    
    # 加载 .env 文件
    load_dotenv()

    # 获取数据库相对路径并基于 PROJECT_ROOT 拼接
    db_path_env = os.getenv("DB_PATH")
    if not db_path_env:
        raise ValueError("未在 .env 中找到 DB_PATH")
    
    # 使用 PROJECT_ROOT 拼接，确保路径正确
    db_full_path = os.path.join(PROJECT_ROOT, db_path_env)
    
    print(f"数据库路径: {db_full_path}")
    print(f"数据库文件存在: {os.path.exists(db_full_path)}")
    
    stock_table = os.getenv("TABLE_STOCK_DAILY")
    if not stock_table:
        raise ValueError("未在 .env 中找到 TABLE_STOCK_DAILY")

    final_query = query_template.format(table_name=stock_table)

    # 连接数据库
    conn = sqlite3.connect(db_full_path)
    try:
        df = pd.read_sql_query(final_query, conn)
        return df
    finally:
        conn.close()


In [4]:
def sql_conn(query_template: str) -> pd.DataFrame:
    """
    连接数据库并执行查询
    """
    import sqlite3
    
    # 加载 .env 文件
    load_dotenv()

    # 获取数据库相对路径并基于 PROJECT_ROOT 拼接
    db_path_env = os.getenv("DB_PATH")
    if not db_path_env:
        raise ValueError("未在 .env 中找到 DB_PATH")
    
    # 使用 PROJECT_ROOT 拼接，确保路径正确
    db_full_path = os.path.join(PROJECT_ROOT, db_path_env)
    
    print(f"数据库路径: {db_full_path}")
    print(f"数据库文件存在: {os.path.exists(db_full_path)}")
    
    stock_table = os.getenv("TABLE_STOCK_DAILY")
    if not stock_table:
        raise ValueError("未在 .env 中找到 TABLE_STOCK_DAILY")

    final_query = query_template.format(table_name=stock_table)

    # 连接数据库
    conn = sqlite3.connect(db_full_path)
    try:
        df = pd.read_sql_query(final_query, conn)
        return df
    finally:
        conn.close()
# --- 执行查询 ---
query = """
SELECT ts_code, trade_date, open, close, vol, low,high
FROM {table_name}
WHERE trade_date BETWEEN '20240101' AND  '20241001'
    AND open IS NOT NULL
    AND close IS NOT NULL
    AND vol IS NOT NULL
    AND vol > 0
"""

df = sql_conn(query)
print(f"查询结果: {len(df)} 条记录")
df['trade_date']= pd.to_datetime(df['trade_date'])
df.head(10)



数据库路径: /Applications/obsidian repo/Obsidian Repo/Finance/Investment/Alpha101/data/db/stock_data.db
数据库文件存在: True
查询结果: 943586 条记录


,ts_code,trade_date,open,close,vol,low,high
0,600000.SH,2024-01-02,6.63,6.60,220667.00,6.60,6.65
1,600519.SH,2024-01-02,1715.00,1685.01,32156.44,1678.10,1718.19
2,000001.SZ,2024-01-02,9.39,9.21,1158366.45,9.21,9.42
3,000002.SZ,2024-01-02,10.44,10.15,811106.29,10.15,10.48
4,300001.SZ,2024-01-02,20.02,20.98,271285.81,20.01,21.42
5,300750.SZ,2024-01-02,162.22,156.83,214032.68,156.62,162.53
6,000004.SZ,2024-01-02,16.10,16.14,28867.00,16.05,16.44
7,000006.SZ,2024-01-02,4.58,4.47,261947.19,4.45,4.60
8,000007.SZ,2024-01-02,4.87,4.84,7274.50,4.80,4.89
9,000008.SZ,2024-01-02,2.36,2.38,217665.19,2.34,2.39


In [ ]:
df['trade_date'].unique()

In [ ]:
open_df = df.pivot(index='trade_date',columns='ts_code',values='open')
close_df = df.pivot(index='trade_date', columns='ts_code', values='close')
vol_df = df.pivot(index='trade_date', columns='ts_code', values='vol')
vol_df.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha002
alpha002 = calculateAlpha002(open_price=open_df,close_price=close_df,volume=vol_df)


In [ ]:
alpha002.dropna(how='all', inplace=True)
alpha002.head(10)

回测：

获取每日alpha002为前10的股票名称

In [ ]:
import pandas as pd
def get_long_short_portfolios(df: pd.DataFrame, n:int =10)->dict:
    """
    获取每日因子值前 n (Long) 和后 n (Short) 的股票代码
    """
    # 提取多头：因子值最大的 n 个
    long_list = df.apply(lambda x: x.nlargest(n).index.tolist(), axis=1)
    
    # 提取空头：因子值最小的 n 个
    short_list = df.apply(lambda x: x.nsmallest(n).index.tolist(), axis=1)
    
    # 组装成一个结果表
    portfolio = pd.DataFrame({
        'long_picks': long_list,
        'short_picks': short_list
    })
    
    return portfolio

In [ ]:
alpha002_long_short = get_long_short_portfolios(df=alpha002,n=10)

alpha002_long_short['long_picks']

Alpha 003

In [ ]:
from core.alpha_factors import calculateAlpha003
alpha003 = calculateAlpha003(open_price=open_df,volume=vol_df)
alpha003.head(10)


In [ ]:
alpha003.dropna(how='all',inplace=True)
alpha003

Alpha4

In [ ]:
low_df = df.pivot(index='trade_date',columns='ts_code',values='low')
# vol_df = df.pivot(index='trade_date',columns='ts_code',values='vol')

low_df.head()

In [ ]:
from core.alpha_factors import calculateAlpha004
alpha004 = calculateAlpha004(low_price=low_df)
alpha004.head(10)

In [ ]:
alpha004.dropna(how='all', inplace=True)
alpha004.head(10)

Alpha 5

In [ ]:
from core.alpha_factors import calculateAlpha005
alpha005 = calculateAlpha005(open_price=open_df,close_price=close_df)
alpha005.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha006
alpha006 = calculateAlpha006(open_price=open_df,close_price=close_df)
alpha006.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha007
alpha007 = calculateAlpha007(open_price=open_df,close_price=close_df)
alpha007.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha008
alpha008 = calculateAlpha008(open_price=open_df,close_price=close_df)
alpha008.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha009
alpha009 = calculateAlpha009(open_price=open_df,close_price=close_df)
alpha009.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha010
alpha010 = calculateAlpha010(open_price=open_df,close_price=close_df)
alpha010.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha011
alpha011 = calculateAlpha011(open_price=open_df,close_price=close_df)
alpha011.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha012
alpha012 = calculateAlpha005(open_price=open_df,close_price=close_df)
alpha012.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha013
alpha013 = calculateAlpha005(open_price=open_df,close_price=close_df)
alpha013.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha014
alpha014 = calculateAlpha014(open_price=open_df,close_price=close_df)
alpha014.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha015
alpha015 = calculateAlpha015(open_price=open_df,close_price=close_df)
alpha015.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha016
alpha016 = calculateAlpha016(open_price=open_df,close_price=close_df)
alpha016.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha017
alpha017 = calculateAlpha017(open_price=open_df,close_price=close_df)
alpha017.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha018
alpha018 = calculateAlpha018(open_price=open_df,close_price=close_df)
alpha018.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha019
alpha019 = calculateAlpha019(open_price=open_df,close_price=close_df)
alpha019.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha020
alpha020 = calculateAlpha020(open_price=open_df,close_price=close_df)
alpha020.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha021
alpha021 = calculateAlpha021(open_price=open_df,close_price=close_df)
alpha021.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha022
alpha022 = calculateAlpha022(open_price=open_df,close_price=close_df)
alpha022.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha023
alpha023 = calculateAlpha023(open_price=open_df,close_price=close_df)
alpha023.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha024
alpha024 = calculateAlpha024(open_price=open_df,close_price=close_df)
alpha024.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)

In [ ]:
from core.alpha_factors import calculateAlpha025
alpha025 = calculateAlpha025(open_price=open_df,close_price=close_df)
alpha025.head(10)